In [1]:
# Import necessary libraries
import json, itertools
import torch
import torch.nn.functional as F
from transformers import BertForMaskedLM, BertTokenizer
from tqdm.auto import tqdm

c:\Users\catha\Documents\agentic-protein-reconstruction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the pretrained ProtBERT MLM model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # set device to cpu or gpu if available
tokeniser = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
mlm = BertForMaskedLM.from_pretrained("Rostlab/prot_bert").to(device)
mlm.eval() # evaluation mode
print(f"Device: {device}")

Loading weights: 100%|██████████| 492/492 [00:00<00:00, 1023.01it/s, Materializing param=cls.predictions.transform.dense.weight]                 
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Rostlab/prot_bert
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Device: cuda


In [3]:
# Load a sample fragmented protein
with open("../data/fragmented_ecoli.jsonl") as f:
    sample = json.loads(f.readline())

fragments = sample["fragments"]
original = sample["ecoli_original"]
print(f"Original Protein: {original}")
print(f"Original Length: {len(original)}")
print(f"Fragmented Protein: {fragments}")
print(f"Number of Fragments: {len(fragments)}")

Original Protein: MSQPITRENFDEWMIPVYAPAPFIPVRGEGSRLWDQQGKEYIDFAGGIAVNALGHAHPELREALNEQASKFWHTGNGYTNESVLRLAKKLIDATFADRVFFCNSGAEANEAALKLARKFAHDRYGSHKSGIVAFKNAFHGRTLFTVSAGGQPAYSQDFAPLPPDIRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRTGELYAYMHYGVTPDLLTTAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKQRHDWFVERLNIINHRYGLFNEVRGLGLLIGCVLNADYAGQAKQISQEAAKAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRFAVACEHFVSRGSS
Original Length: 406
Fragmented Protein: ['GSS', 'LAR', 'EYIDFAGGIAVNALGHAHPELR', 'NAFHGR', 'FWHTGNGYTNESVLR', 'K', 'HNALLIFDEVQTGVGR', 'LWDQQGK', 'QISQEAAK', 'FAHDR', 'ENFDEWMIPVYAPAPFIPVR', 'HAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLR', 'SGIVAFK', 'FAVACEHFVSR', 'VFFCNSGAEANEAALK', 'FAPALNVSEEEVTTGLDR', 'GEGSR', 'TLFTVSAGGQPAYSQDFAPLPPDIR', 'LIDATFADR', 'VLELINTPEMLNGVK', 'EALNEQASK', 'ELCDR', 'YGLFNEVR', 'YGSHK', 'AGVMVLIAGGNVVR', 'K', 'HDWFVER', 'LNIINHR', 'TGELYAYMHYGVTPDLLTTAK', 'GLGLLIGCVLNADYAGQAK', 'ALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGK', 'MSQPITR', 'QR'

In [4]:
def score_junctions(fragments, batch_size=32):
    """Score all pairwise fragment junctions using batched ProtBERT MLM."""
    n = len(fragments)
    pairs = list(itertools.permutations(range(n), 2)) # all ordered pairs of fragments
    inputs = [] # sequences with a [MASK] at the junction
    targets = [] # first residue of the next fragment
    for i, j in pairs:
        a, b = fragments[i], fragments[j]
        targets.append(b[0]) # we want to predict this
        inputs.append(" ".join(list(a)) + " [MASK] " + " ".join(list(b[1:]))) # put a mask between the fragments
    mat = torch.zeros(n, n) # score matrix
    for start in tqdm(range(0, len(pairs), batch_size), desc="Scoring Junctions"):
        end = min(start + batch_size, len(pairs))
        batch_inputs = tokeniser(inputs[start:end], return_tensors="pt", padding=True, truncation=True) # process in batches
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()} # move to gpu if available
        with torch.no_grad():
            logits = mlm(**batch_inputs).logits # [batch_size, seq_len, vocab_size]
        for k, (i, j) in enumerate(pairs[start:end]):
            mask_index = (batch_inputs["input_ids"][k] == tokeniser.mask_token_id).nonzero(as_tuple=False)[0, 0]
            probs = F.softmax(logits[k, mask_index], dim=-1)
            mat[i, j] = probs[tokeniser.convert_tokens_to_ids(targets[start + k])].item()
    return mat


def greedy_order(scores: torch.Tensor) -> list[int]:
    """Greedily pick the next fragment with highest score from current fragment."""
    n = scores.shape[0]
    used = set()
    start = scores.sum(dim=0).argmin().item() # start with fragment least likely to be after any other
    order = [start]
    used.add(start)
    while len(order) < n:
        row = scores[order[-1]].clone() # consider all candidates after the last fragment in order
        row[list(used)] = -1 # ignore already used fragments
        next = row.argmax().item() # pick highest probability
        order.append(next)
        used.add(next)
    return order

In [6]:
# Reconstruct the sample
scores = score_junctions(fragments) # score the junctions
order = greedy_order(scores) # greedy ordering
result = "".join(fragments[i] for i in order)
print(f"Original:      {original}")
print(f"Reconstructed: {result}")
print(f"Match:         {result == original}")

Scoring Junctions: 100%|██████████| 36/36 [00:09<00:00,  4.00it/s]

Original:      MSQPITRENFDEWMIPVYAPAPFIPVRGEGSRLWDQQGKEYIDFAGGIAVNALGHAHPELREALNEQASKFWHTGNGYTNESVLRLAKKLIDATFADRVFFCNSGAEANEAALKLARKFAHDRYGSHKSGIVAFKNAFHGRTLFTVSAGGQPAYSQDFAPLPPDIRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRTGELYAYMHYGVTPDLLTTAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKQRHDWFVERLNIINHRYGLFNEVRGLGLLIGCVLNADYAGQAKQISQEAAKAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRFAVACEHFVSRGSS
Reconstructed: MSQPITRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRVFFCNSGAEANEAALKLAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKELCDRLIDATFADRLARLWDQQGKGEGSRGLGLLIGCVLNADYAGQAKEYIDFAGGIAVNALGHAHPELREALNEQASKENFDEWMIPVYAPAPFIPVRTGELYAYMHYGVTPDLLTTAKSGIVAFKKLNIINHRKTLFTVSAGGQPAYSQDFAPLPPDIRQISQEAAKHNALLIFDEVQTGVGRHDWFVERFAHDRFAVACEHFVSRFWHTGNGYTNESVLRNAFHGRGSSQRYGLFNEVRYGSHK
Match:         False
